# MCP4RS Reproducible Media Gallery Demo

This Colab demonstrates the media pipeline independently from the main MCP4RS app/server. The optional Hugging Face `app.py` in this repo exposes the same workflow as buttons.

The goal is to prove the gallery can be regenerated from open-data source queries and Python render scripts, without committing the media files in advance.

`generated/provenance/` records original source URLs, STAC item IDs, WMS URLs, and asset links. `media/` contains final generated gallery outputs. `figures/` contains processed intermediate figures and frames, not original-source figures.

Workflow:

1. Clone the standalone media demo repo.
2. Install geospatial/media dependencies.
3. Export app-independent source URLs and STAC asset links that mirror the MCP4RS source-discovery tools.
4. Generate the Mermaid architecture source and a smoke-test media gallery.
5. Inspect generated images, GIFs, intermediate figures, and provenance JSON.

For the first demonstration, run the smoke test with `--skip-long --continue-on-error`. The full GIF time-series generation can take longer.


In [ ]:
# @title 1. Clone the standalone demo repo
REPO_URL = "https://github.com/MCP4RemoteSensing/mcp4rs-media-gallery.git"  # @param {type:"string"}
WORKDIR = "mcp4rs-media-gallery"  # @param {type:"string"}

import os
import pathlib
import subprocess

workdir = pathlib.Path(WORKDIR)
if workdir.exists():
    print(f"Using existing folder: {workdir}")
else:
    print(f"Cloning {REPO_URL}")
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)

os.chdir(workdir)
print("Current directory:", pathlib.Path.cwd())
print("Top-level files:")
for path in sorted(pathlib.Path(".").iterdir()):
    print("-", path)

If the GitHub repo has not been created yet, push the local folder first using the commands in `docs/GITHUB_REPO_SETUP.md`, then reopen this notebook from GitHub in Colab.

In [ ]:
# @title 2. Install dependencies
%pip -q install -r requirements.txt

In [ ]:
# Confirm that media outputs are not pre-shipped.
from pathlib import Path

for folder in ["media", "figures", "generated/provenance"]:
    paths = sorted(p.name for p in Path(folder).iterdir())
    print(folder, "->", paths)

## Step 1: Export source URLs

This step calls `source_queries.py`, which mirrors the source-discovery behavior of the main MCP4RS tools: `search_open_data`, `search_catalog`, and `get_nightlights`. It records returned source URLs and STAC asset links before any media rendering happens.

In [ ]:
!python scripts/export_media_sources.py

In [ ]:
import json
from pathlib import Path
from pprint import pprint

manifest_path = Path("generated/provenance/media_sources.json")
manifest = json.loads(manifest_path.read_text())
print("Manifest:", manifest_path)
print("Generated at:", manifest.get("generated_at"))
print()

for record in manifest.get("mcp_tool_records", []):
    key = record.get("key")
    tool = record.get("tool")
    count = record.get("count")
    error = record.get("error")
    image_url = record.get("image_url")
    items = record.get("items") or []
    first_item = items[0] if items else {}
    asset_keys = sorted((first_item.get("assets") or {}).keys())
    print(f"{key} | tool={tool} | count={count} | assets={asset_keys}")
    if image_url:
        print("  image_url:", image_url[:180] + ("..." if len(image_url) > 180 else ""))
    if error:
        print("  error:", error)

print("\nExample nightlights record:")
nightlights = next((r for r in manifest["mcp_tool_records"] if r.get("key") == "nightlights_prd_png_source"), None)
pprint(nightlights)

print("\nExample Sentinel-2 asset URLs:")
s2 = next((r for r in manifest["mcp_tool_records"] if r.get("key") == "s2_workflow_and_water_fraction"), None)
if s2 and s2.get("items"):
    pprint(s2["items"][0].get("assets"))

## Step 2: Generate the media gallery

The smoke test skips the long lake/desert time-series animations but still tests the full source-query -> render -> media-output pattern. It also writes `media/architecture.mmd`, a Mermaid source file for the architecture diagram. `--continue-on-error` is useful in Colab because public geospatial endpoints can be temporarily slow or unavailable.


In [ ]:
# @title 3. Generate a smoke-test gallery
!python scripts/generate_media_gallery.py --skip-long --continue-on-error

In [ ]:
from IPython.display import Image as IPImage, Markdown, display
from pathlib import Path

architecture_path = Path("media/architecture.mmd")
if architecture_path.exists():
    print("Generated Mermaid architecture:")
    display(Markdown("```mermaid\n" + architecture_path.read_text() + "\n```"))

media_paths = sorted(list(Path("media").glob("*.png")) + list(Path("media").glob("*.gif")))
figure_paths = sorted(Path("figures").glob("*.png"))

print("Generated final media outputs:")
for path in media_paths:
    print("-", path)
    display(IPImage(filename=str(path)))

print("Processed intermediate figure outputs:")
for path in figure_paths:
    print("-", path)


In [ ]:
summary_path = Path("generated/provenance/generated_media_summary.json")
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    pprint(summary)

print("\nProvenance files:")
for path in sorted(Path("generated/provenance").glob("*.json")):
    print("-", path)

## Optional: Full gallery generation

The full run also creates the longer `desert_greening.gif`, `lopnur_ponds.gif`, and `hongjiannao_lake.gif`. It may take much longer because it reads multiple cloud-optimized GeoTIFF scenes.

In [ ]:
# @title 4. Optional full generation
RUN_FULL_GALLERY = False  # @param {type:"boolean"}

import subprocess

if RUN_FULL_GALLERY:
    subprocess.run(["python", "scripts/generate_media_gallery.py", "--continue-on-error"], check=True)
else:
    print("Full gallery skipped. Set RUN_FULL_GALLERY=True to run it.")

In [ ]:
# @title 5. Optional: package generated outputs
CREATE_OUTPUT_ZIP = True  # @param {type:"boolean"}

import subprocess
from pathlib import Path

if CREATE_OUTPUT_ZIP:
    out_zip = Path("mcp4rs-generated-media.zip")
    if out_zip.exists():
        out_zip.unlink()
    subprocess.run(["zip", "-qr", str(out_zip), "media", "figures", "generated/provenance"], check=True)
    print("Created", out_zip.resolve())
else:
    print("Output zip skipped.")

## Next step

After this Colab workflow is stable, the same workflow can be wrapped as an Agent Skill. The skill can call the main MCP4RS tools, save returned source metadata, run the render scripts, and return the gallery plus provenance. That integration is intentionally outside this standalone demo.

A future Hugging Face Space can expose the same flow through buttons: export source provenance, generate the gallery, show the Mermaid architecture, display processed media, and offer the generated files for download.
